# 9. Model Selection & Evaluation

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, learning_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, mean_squared_error, r2_score, classification_report
from sklearn.datasets import load_iris, load_diabetes, make_classification
from sklearn.svm import SVC

## 9.1 Train/Test Split

**Train/Test Split** is the simplest approach to model evaluation. We split the dataset into a training set and a testing set. The model is trained on the training set and evaluated on the testing set.

sklearn.**train_test_split**(*arrays, test_size=0.25, random_state=None*)

In [2]:
iris = load_iris()
X, y = iris.data, iris.target
print ('Dataset size: %d samples, %d features\n'%(X.shape[0], X.shape[1]))
print ('Class labels: %s\n'%str(np.unique(y)))

Dataset size: 150 samples, 4 features
Class labels: [0 1 2]


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print ('X_train shape: %s\n'%str(X_train.shape))
print ('X_test shape: %s\n'%str(X_test.shape))
print ('y_train shape: %s\n'%str(y_train.shape))
print ('y_test shape: %s\n'%str(y_test.shape))

X_train shape: (112, 4)
X_test shape: (38, 4)
y_train shape: (112,)
y_test shape: (38,)


<hr>

## 9.2 Cross-Validation

**Cross-Validation** divides the data into k folds. The model is trained on k-1 folds and validated on the remaining fold. This process repeats k times.

sklearn.**cross_val_score**(*estimator, X, y, cv=5*)

#### 9.2.1 Cross-Validation with cv=5

In [4]:
model = SVC(kernel='rbf', random_state=42)
scores = cross_val_score(model, X, y, cv=5)
print ('Cross-validation scores (cv=5): %s\n'%str(scores))
print ('Mean accuracy: %.4f\n'%scores.mean())
print ('Standard deviation: %.4f\n'%scores.std())

Cross-validation scores (cv=5): [0.96666667 1.         0.96666667 0.96666667 1.        ]
Mean accuracy: 0.9800
Standard deviation: 0.0163


#### 9.2.2 Cross-Validation with cv=10

In [5]:
scores_10 = cross_val_score(model, X, y, cv=10)
print ('Cross-validation scores (cv=10): %s\n'%str(scores_10))
print ('Mean accuracy: %.4f\n'%scores_10.mean())
print ('Standard deviation: %.4f\n'%scores_10.std())

Cross-validation scores (cv=10): [1.         1.         0.93333333 1.         0.93333333 0.93333333
 1.         1.         0.93333333 1.        ]
Mean accuracy: 0.9733
Standard deviation: 0.0327


<hr>

## 9.3 Grid Search CV

**Grid Search** exhaustively searches over a specified parameter grid to find the best hyperparameters.

sklearn.**GridSearchCV**(*estimator, param_grid, cv=5, scoring='accuracy'*)

In [6]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1],
    'kernel': ['rbf']
}
grid_search = GridSearchCV(SVC(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X, y)

In [7]:
print ('Best parameters: %s\n'%str(grid_search.best_params_))
print ('Best cross-validation score: %.4f\n'%grid_search.best_score_)
print ('\nGrid search results:')
results = grid_search.cv_results_
for i in range(len(results['params'])):
    print ('C=%s, gamma=%s, score=%.4f'%(results['params'][i]['C'], results['params'][i]['gamma'], results['mean_test_score'][i]))

Best parameters: {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
Best cross-validation score: 0.9867

Grid search results:
C=0.1, gamma=0.001, score=0.9467
C=0.1, gamma=0.01, score=0.9467
C=0.1, gamma=0.1, score=0.9467
C=0.1, gamma=1, score=0.9467
C=1, gamma=0.001, score=0.9600
C=1, gamma=0.01, score=0.9733
C=1, gamma=0.1, score=0.9867
C=1, gamma=1, score=0.9733
C=10, gamma=0.001, score=0.9733
C=10, gamma=0.01, score=0.9867
C=10, gamma=0.1, score=0.9800
C=10, gamma=1, score=0.9733
C=100, gamma=0.001, score=0.9733
C=100, gamma=0.01, score=0.9800
C=100, gamma=0.1, score=0.9800
C=100, gamma=1, score=0.9733


<hr>

## 9.4 Randomized Search CV

**Randomized Search** samples a fixed number of hyperparameter combinations from specified distributions. It is more efficient than Grid Search for high-dimensional spaces.

sklearn.**RandomizedSearchCV**(*estimator, param_distributions, n_iter=10, cv=5*)

In [8]:
param_dist = {
    'C': np.logspace(-2, 2, 10),
    'gamma': np.logspace(-3, 1, 10),
    'kernel': ['rbf']
}
random_search = RandomizedSearchCV(SVC(random_state=42), param_dist, n_iter=10, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
random_search.fit(X, y)

In [9]:
print ('Randomized Search best parameters: %s\n'%str(random_search.best_params_))
print ('Best cross-validation score: %.4f\n'%random_search.best_score_)
print ('\nRandom search results:')
results_rs = random_search.cv_results_
for i in range(len(results_rs['params'])):
    print ('Iteration %d: C=%.2f, gamma=%.2f, score=%.4f'%(i+1, results_rs['params'][i]['C'], results_rs['params'][i]['gamma'], results_rs['mean_test_score'][i]))

Randomized Search best parameters: {'kernel': 'rbf', 'gamma': 0.16681005372000587, 'C': 1.6681005372000588}
Best cross-validation score: 0.9867

Random search results:
Iteration 1: C=1.67, gamma=0.17, score=0.9867
Iteration 2: C=0.46, gamma=0.02, score=0.9667
Iteration 3: C=7.74, gamma=0.17, score=0.9800
Iteration 4: C=0.46, gamma=1.67, score=0.9733
Iteration 5: C=7.74, gamma=0.46, score=0.9733
Iteration 6: C=0.10, gamma=0.46, score=0.9467
Iteration 7: C=35.94, gamma=0.05, score=0.9800
Iteration 8: C=1.67, gamma=0.46, score=0.9800
Iteration 9: C=35.94, gamma=0.46, score=0.9733
Iteration 10: C=0.10, gamma=0.05, score=0.9600


<hr>

## 9.5 Evaluation Metrics

Different tasks require different evaluation metrics. Below we cover metrics for both **classification** and **regression**.

#### 9.5.1 Classification Metrics

sklearn.**accuracy_score**(*y_true, y_pred*)

sklearn.**precision_score**(*y_true, y_pred, average='macro'*)

sklearn.**recall_score**(*y_true, y_pred, average='macro'*)

sklearn.**f1_score**(*y_true, y_pred, average='macro*')

sklearn.**confusion_matrix**(*y_true, y_pred*)

In [10]:
X_c, y_c = make_classification(n_samples=200, n_features=10, n_informative=5, n_classes=2, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.3, random_state=42)

clf = SVC(kernel='rbf', random_state=42)
clf.fit(X_train_c, y_train_c)
y_pred_c = clf.predict(X_test_c)

print ('Classification Report:')
print (classification_report(y_test_c, y_pred_c))

print ('Accuracy: %.4f\n'%accuracy_score(y_test_c, y_pred_c))
print ('Precision (macro): %.4f\n'%precision_score(y_test_c, y_pred_c, average='macro'))
print ('Recall (macro): %.4f\n'%recall_score(y_test_c, y_pred_c, average='macro'))
print ('F1-Score (macro): %.4f\n'%f1_score(y_test_c, y_pred_c, average='macro'))
print ('\nConfusion Matrix:')
print (confusion_matrix(y_test_c, y_pred_c))

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        13
           2       1.00      1.00      1.00        19

    accuracy                           1.00        51
   macro avg       1.00      1.00      1.00        51
weighted avg       1.00      1.00      1.00        51


Accuracy: 1.0000
Precision (macro): 1.0000
Recall (macro): 1.0000
F1-Score (macro): 1.0000

Confusion Matrix:
 [[19  0  0]
 [ 0 13  0]
 [ 0  0 19]]


#### 9.5.2 Regression Metrics

sklearn.**mean_squared_error**(*y_true, y_pred*)

sklearn.**r2_score**(*y_true, y_pred*)

In [11]:
from sklearn.linear_model import LinearRegression

diabetes = load_diabetes()
X_d, y_d = diabetes.data, diabetes.target
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_d, y_d, test_size=0.25, random_state=42)

lr = LinearRegression()
lr.fit(X_train_d, y_train_d)
y_pred_d = lr.predict(X_test_d)

mse = mean_squared_error(y_test_d, y_pred_d)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_d, y_pred_d)

print ('Regression Metrics:')
print ('Mean Squared Error: %.4f\n'%mse)
print ('Root Mean Squared Error: %.4f\n'%rmse)
print ('R2 Score: %.4f\n'%r2)

Regression Metrics:
Mean Squared Error: 2882.6158
Root Mean Squared Error: 53.6899
R2 Score: 0.5143


<hr>

## 9.6 Learning Curves

**Learning Curves** plot training and validation scores as a function of training set size. They help diagnose **bias** and **variance** problems.

#### 9.6.1 Bias (Underfitting)
A model with **high bias** is too simple. Both training and validation scores are low and converge to a poor value. Adding more data does not help.

#### 9.6.2 Variance (Overfitting)
A model with **high variance** is too complex. The training score is high but the validation score is low, creating a large **generalization gap**. Adding more data can help reduce overfitting.

#### 9.6.3 The Bias-Variance Tradeoff
The goal is to find the sweet spot between underfitting and overfitting. **Regularization**, **cross-validation**, and **more training data** are common strategies.

In [12]:
train_sizes, train_scores, val_scores = learning_curve(
    SVC(kernel='rbf', C=1, gamma=0.1, random_state=42),
    X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 6), random_state=42
)
print ('Computing learning curves...')
print ('Training sizes: %s\n'%str(train_sizes))

Computing learning curves...
Training sizes: [ 25  50  75 100 125 150]


In [13]:
train_mean = np.mean(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, 'o-', label='Training Score')
plt.plot(train_sizes, val_mean, 'o-', label='Validation Score')
plt.xlabel('Training Set Size')
plt.ylabel('Score')
plt.title('Learning Curves (SVM on Iris)')
plt.legend(loc='best')
plt.grid(True)
plt.show()

<hr>

## 9.7 Assignment

1. Load the **digits** dataset from sklearn.datasets.
2. Split the data into train and test sets (80/20 split with random_state=42).
3. Train an **SVC** with default parameters and report accuracy on the test set.
4. Use **GridSearchCV** to tune `C` and `gamma` for the SVM. Report best parameters and best score.
5. Use **RandomizedSearchCV** with n_iter=10 to find hyperparameters. Compare results.
6. Compute and print: accuracy, precision (macro), recall (macro), and F1-score (macro).
7. Plot learning curves for the SVM on the digits dataset.
8. Discuss bias-variance tradeoff observed in your learning curves.